 Reject/accept the 4 null hypotheses with p-values + business impact


In [7]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import os
import subprocess

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Auto-restore from DVC if missing
data_path = '../data/raw_insurance_data.csv'
if not os.path.exists(data_path):
    print("Data missing — restoring from DVC...")
    subprocess.run(['dvc', 'checkout'], check=True)
    if not os.path.exists(data_path):
        raise FileNotFoundError(f"Data not found at {data_path}. Re-download and add to DVC.")

print(f"Loading from: {os.path.abspath(data_path)}")

# IMPORTANT COLS ONLY to save memory
important_cols = ['Province', 'PostalCode', 'Gender', 'TotalPremium', 'TotalClaims']

# CHUNKED LOADING: Process in 100k row chunks to avoid OOM
chunk_size = 100000
agg_df = pd.DataFrame()  # To accumulate aggregates

for chunk in pd.read_csv(data_path, sep='|', usecols=important_cols, chunksize=chunk_size, low_memory=False, on_bad_lines='skip'):
    # Clean numeric in chunk
    chunk['TotalPremium'] = pd.to_numeric(chunk['TotalPremium'], errors='coerce').fillna(0)
    chunk['TotalClaims'] = pd.to_numeric(chunk['TotalClaims'], errors='coerce').fillna(0)
    
    # Compute metrics in chunk
    chunk['HasClaim'] = (chunk['TotalClaims'] > 0).astype(int)
    chunk['Margin'] = chunk['TotalPremium'] - chunk['TotalClaims']
    chunk['LossRatio'] = np.where(chunk['TotalPremium'] > 0, chunk['TotalClaims'] / chunk['TotalPremium'], 0)
    
    # Append to agg_df (only needed for full df; for aggregates, we can sum here)
    agg_df = pd.concat([agg_df, chunk], ignore_index=True)

# If agg_df is still too big, use dask for out-of-memory
try:
    import dask.dataframe as dd
    print("Using Dask for out-of-memory processing...")
    dd_df = dd.read_csv(data_path, sep='|', usecols=important_cols)
    df = dd_df.compute()  # Compute only when needed
except ImportError:
    print("Dask not installed — using pandas chunks (may use more RAM)")

# Final df is now ready (or use agg_df for aggregates)
print(f"Loaded successfully! Shape: {agg_df.shape}")
print("\nSample data:")
print(agg_df.head())
print(f"\nUnique Provinces: {agg_df['Province'].nunique()} ({agg_df['Province'].unique()})")
print(f"Overall Loss Ratio: {agg_df['LossRatio'].mean():.2%}")

# Now continue with the rest of your notebook (groupbys, tests, plots)
# ...

Loading from: c:\Users\bezaw\OneDrive\Desktop\10Acadamy-KAIM\Insurance\Week3-Insurance\data\raw_insurance_data.csv
Dask not installed — using pandas chunks (may use more RAM)
Loaded successfully! Shape: (1000098, 8)

Sample data:
          Gender Province  PostalCode  TotalPremium  TotalClaims  HasClaim  \
0  Not specified  Gauteng        1459     21.929825          0.0         0   
1  Not specified  Gauteng        1459     21.929825          0.0         0   
2  Not specified  Gauteng        1459      0.000000          0.0         0   
3  Not specified  Gauteng        1459    512.848070          0.0         0   
4  Not specified  Gauteng        1459      0.000000          0.0         0   

       Margin  LossRatio  
0   21.929825        0.0  
1   21.929825        0.0  
2    0.000000        0.0  
3  512.848070        0.0  
4    0.000000        0.0  

Unique Provinces: 9 (['Gauteng' 'KwaZulu-Natal' 'Mpumalanga' 'Eastern Cape' 'Western Cape'
 'Limpopo' 'North West' 'Free State' 'Northern 